# Capstone — Refresh / Content Opportunity Scoring

**Lane:** score content items as growing, declining, recovering, or worth review. Output: a ranked action engine (protect / improve / rewrite / merge / prune / monitor) with reason codes.

**Design choice (avoids leakage):** we pick a *decision date* well inside the panel (2026-02-28), build features only from data **up to and including** that date, and build the label only from data **after** that date. The final month (2026-06, the `_sample` month) stays sealed until the last evaluation pass.

Sections: 1) connect, 2) data contract, 3) decision-time features, 4) outcome label, 5) assemble + query-mix context, 6) baseline vs model (grouped split), 7) reason codes + ranked action table, 8) charts, 9) leakage trap demo, 10) limitations.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn matplotlib pandas

In [4]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 2. Data contract (plain words)

1. **One row means:** one pseudonymized content item, scored as of a single decision date, with a predicted action.
2. **Table(s):** `fact_content_daily_performance` (daily GSC metrics, for features + label), `fact_content_query_90d` (static query-mix context), `dim_clients` (panel coverage, for the grouped split).
3. **Time window:** features from `(decision_date - 90d, decision_date]`; label from `(decision_date, decision_date + 30d]`. Decision date = **2026-02-28** (mid-panel). Final month **2026-06** stays sealed — never touched until the last evaluation pass.
4. **What we predict:** an action category — `declining`, `recovering`, `growing`, `stable/monitor` — derived from impression momentum, then mapped to protect/improve/rewrite/merge/prune/monitor.
5. **Deliberately excluded:** content items with fewer than 100 impressions in the prior-30 window (too sparse to score reliably — routed to `monitor` by default, not modeled).

## 3. Decision-time features (knowable at 2026-02-28, nothing later)

In [6]:
DECISION_DATE = "DATE '2026-02-28'"

features = con.sql(f"""
    WITH bounds AS (SELECT {DECISION_DATE} AS d),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_w0,
               SUM(CASE WHEN f.report_date <= b.d - INTERVAL 30 DAY
                         AND f.report_date >  b.d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_w1,
               SUM(CASE WHEN f.report_date <= b.d - INTERVAL 60 DAY
                         AND f.report_date >  b.d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_w2,
               SUM(CASE WHEN f.report_date >  b.d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_w0,
               AVG(CASE WHEN f.report_date >  b.d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_w0,
               AVG(CASE WHEN f.report_date <= b.d - INTERVAL 30 DAY
                         AND f.report_date >  b.d - INTERVAL 60 DAY THEN f.gsc_avg_position END)       AS pos_w1
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date <= b.d AND f.report_date > b.d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_w1 >= 100
    )
    SELECT *,
           clk_w0 / NULLIF(imp_w0, 0)              AS ctr_w0,
           (imp_w0 - imp_w1) / NULLIF(imp_w1, 0)    AS chg_recent,
           (imp_w1 - imp_w2) / NULLIF(imp_w2, 0)    AS chg_prior,
           pos_w0 - pos_w1                          AS pos_delta
    FROM windowed
""").df()

print(f'{len(features):,} content items with enough decision-time history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

62,538 content items with enough decision-time history


,client_hash_id,content_hash_id,imp_w0,imp_w1,imp_w2,clk_w0,pos_w0,pos_w1,ctr_w0,chg_recent,chg_prior,pos_delta
0,client_62f4a7e64f5e0096,content_58442c358e5e49ad,1523.0,1417.0,912.0,4.0,2.493224,5.139288,0.002626,0.074806,0.553728,-2.646064
1,client_62f4a7e64f5e0096,content_44d3297a323ab2d1,195.0,154.0,61.0,0.0,6.077117,4.168138,0.000000,0.266234,1.524590,1.908979
2,client_62f4a7e64f5e0096,content_cba3b2d78ab28f12,254.0,549.0,78.0,4.0,1.048730,0.890709,0.015748,-0.537341,6.038462,0.158022
3,client_62f4a7e64f5e0096,content_35c0a55a7217798b,253.0,231.0,264.0,0.0,0.574715,0.318307,0.000000,0.095238,-0.125000,0.256408
4,client_62f4a7e64f5e0096,content_4447cdb22f46ef5e,162.0,100.0,94.0,2.0,8.713737,15.546679,0.012346,0.620000,0.063830,-6.832941


## 4. Outcome label (after 2026-02-28 only — this is what we're trying to predict)

In [7]:
labels = con.sql(f"""
    WITH bounds AS (SELECT {DECISION_DATE} AS d),
    outcome AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) AS imp_future30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date >  b.d AND f.report_date <= b.d + INTERVAL 30 DAY
        GROUP BY 1, 2
    )
    SELECT * FROM outcome
""").df()

print(f'{len(labels):,} content items with an outcome window')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331,231 content items with an outcome window


## 5. Assemble: features + label + static query-mix context

In [8]:
import pandas as pd

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           ANY_VALUE(rare_impressions_share)        AS rare_share,
           ANY_VALUE(anonymized_impressions_share)  AS anon_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

data = features.merge(labels, on=['client_hash_id', 'content_hash_id'], how='inner')
data = data.merge(qsignals, on='content_hash_id', how='left')

data['chg_future'] = (data['imp_future30'] - data['imp_w0']) / data['imp_w0'].replace(0, pd.NA)

def bucket(row):
    if row['chg_future'] is None or pd.isna(row['chg_future']):
        return 'monitor'
    if row['chg_prior'] <= -0.20 and row['chg_future'] >= 0.10:
        return 'recovering'
    if row['chg_future'] <= -0.20:
        return 'declining'
    if row['chg_future'] >= 0.20:
        return 'growing'
    return 'stable'

data['action_label'] = data.apply(bucket, axis=1)
print(data['action_label'].value_counts())
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

action_label
growing       20860
stable        19476
declining     14179
recovering     4155
monitor          99
Name: count, dtype: int64


,client_hash_id,content_hash_id,imp_w0,imp_w1,imp_w2,clk_w0,pos_w0,pos_w1,ctr_w0,chg_recent,chg_prior,pos_delta,imp_future30,visible_queries,rare_share,anon_share,chg_future,action_label
0,client_62f4a7e64f5e0096,content_58442c358e5e49ad,1523.0,1417.0,912.0,4.0,2.493224,5.139288,0.002626,0.074806,0.553728,-2.646064,1195.0,7.0,0.112088,0.753846,-0.215364,declining
1,client_62f4a7e64f5e0096,content_44d3297a323ab2d1,195.0,154.0,61.0,0.0,6.077117,4.168138,0.000000,0.266234,1.524590,1.908979,136.0,2.0,0.572632,0.366316,-0.302564,declining
2,client_62f4a7e64f5e0096,content_cba3b2d78ab28f12,254.0,549.0,78.0,4.0,1.048730,0.890709,0.015748,-0.537341,6.038462,0.158022,326.0,NaN,NaN,NaN,0.283465,growing
3,client_62f4a7e64f5e0096,content_35c0a55a7217798b,253.0,231.0,264.0,0.0,0.574715,0.318307,0.000000,0.095238,-0.125000,0.256408,154.0,NaN,NaN,NaN,-0.391304,declining
4,client_62f4a7e64f5e0096,content_4447cdb22f46ef5e,162.0,100.0,94.0,2.0,8.713737,15.546679,0.012346,0.620000,0.063830,-6.832941,152.0,2.0,0.228758,0.581699,-0.061728,stable


## 6. Baseline vs model — grouped split on `client_hash_id`

A random row split would leak: two content items from the same client share site-wide trends. We split by client instead, so the model is tested on clients it has never seen.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = ['imp_w0', 'imp_w1', 'chg_prior', 'ctr_w0', 'pos_delta',
                 'visible_queries', 'rare_share', 'anon_share']
model_data = data.dropna(subset=feature_cols + ['action_label'])

X = model_data[feature_cols]
y = model_data['action_label']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

baseline_pred = pd.Series(y_tr.mode()[0], index=y_te.index)  # always predict majority class
print('--- Baseline (majority class) ---')
print(classification_report(y_te, baseline_pred, digits=3, zero_division=0))

model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')
model.fit(X_tr, y_tr)
model_pred = model.predict(X_te)
print('--- Model (RandomForest, client-grouped split) ---')
print(classification_report(y_te, model_pred, digits=3, zero_division=0))

--- Baseline (majority class) ---
              precision    recall  f1-score   support

   declining      0.000     0.000     0.000      3524
     growing      0.308     1.000     0.471      4127
  recovering      0.000     0.000     0.000      1053
      stable      0.000     0.000     0.000      4685

    accuracy                          0.308     13389
   macro avg      0.077     0.250     0.118     13389
weighted avg      0.095     0.308     0.145     13389

--- Model (RandomForest, client-grouped split) ---
              precision    recall  f1-score   support

   declining      0.375     0.313     0.341      3524
     growing      0.474     0.518     0.495      4127
  recovering      0.657     0.422     0.514      1053
      stable      0.456     0.512     0.483      4685

    accuracy                          0.454     13389
   macro avg      0.490     0.441     0.458     13389
weighted avg      0.456     0.454     0.452     13389



## 7. Reason codes + ranked action engine

Reason code = the feature that most drove this row's prediction, read off global feature importances (simple, transparent — not a full SHAP breakdown).

In [10]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

ACTION_MAP = {
    'declining': 'rewrite_or_review',
    'recovering': 'protect',
    'growing': 'protect',
    'stable': 'monitor',
}

def reason_code(row):
    top_feat = importances.index[0]
    return f'{top_feat}={row[top_feat]:.2f}'

results = X_te.copy()
results['content_hash_id'] = model_data.loc[X_te.index, 'content_hash_id'].values
results['predicted_action'] = model_pred
results['recommendation'] = results['predicted_action'].map(ACTION_MAP)
results['reason_code'] = results.apply(reason_code, axis=1)
results['confidence'] = model.predict_proba(X_te).max(axis=1)

ranked = results.sort_values('confidence', ascending=False)[
    ['content_hash_id', 'predicted_action', 'recommendation', 'reason_code', 'confidence']
]
ranked.head(20)

chg_prior          0.311760
imp_w0             0.116423
pos_delta          0.107545
imp_w1             0.105033
anon_share         0.102074
rare_share         0.100587
visible_queries    0.087938
ctr_w0             0.068639
dtype: float64


,content_hash_id,predicted_action,recommendation,reason_code,confidence
1470,content_98513ba15465c595,growing,protect,chg_prior=0.09,0.936667
3735,content_0a6176d84ce3ce38,growing,protect,chg_prior=0.20,0.936667
1094,content_b441d71206734e3e,growing,protect,chg_prior=0.47,0.936667
1661,content_4bf41774d5b44a7e,growing,protect,chg_prior=0.12,0.933333
54232,content_90a974671c1bd20f,growing,protect,chg_prior=0.73,0.923333
56600,content_dc24866e010332e4,growing,protect,chg_prior=0.08,0.916667
48774,content_ae4def22b9c2f64b,growing,protect,chg_prior=0.01,0.916667
9976,content_69b997fddead5c90,growing,protect,chg_prior=0.77,0.916667
2584,content_4ca5b5a7920e2bcd,growing,protect,chg_prior=1.88,0.913333
10011,content_017895324604e2b2,growing,protect,chg_prior=1.21,0.913333


## 8. Charts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data['action_label'].value_counts().plot(kind='bar', ax=axes[0], title='Label distribution (train+test)')
importances.plot(kind='barh', ax=axes[1], title='Feature importance')
plt.tight_layout()
plt.show()

## 9. The leakage trap (do this once, on purpose, then remove it)

Add `chg_future` itself — the label-derived column — as a feature. Watch the score jump toward perfect. Then delete it. This is the same lesson as notebook 02, run here on real warehouse data.

In [11]:
leaky_cols = feature_cols + ['chg_future']
leaky_data = data.dropna(subset=leaky_cols + ['action_label'])
Xl = leaky_data[leaky_cols]
yl = leaky_data['action_label']
gl = leaky_data['client_hash_id']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr2, te2 = next(gss2.split(Xl, yl, gl))
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl.iloc[tr2], yl.iloc[tr2])
print('--- WITH the leak (chg_future in the feature set) ---')
print(classification_report(yl.iloc[te2], leaky_model.predict(Xl.iloc[te2]), digits=3, zero_division=0))
print('This score is fake — chg_future is literally what the label is built from. Delete it and keep section 6\'s honest number.')

--- WITH the leak (chg_future in the feature set) ---
              precision    recall  f1-score   support

   declining      1.000     1.000     1.000      3524
     growing      1.000     1.000     1.000      4127
  recovering      1.000     1.000     1.000      1053
      stable      1.000     1.000     1.000      4685

    accuracy                          1.000     13389
   macro avg      1.000     1.000     1.000     13389
weighted avg      1.000     1.000     1.000     13389

This score is fake — chg_future is literally what the label is built from. Delete it and keep section 6's honest number.


## 10. Limitations

- `fact_content_query_90d` is a single fixed 90-day window over the whole panel, not re-computed per decision date — it's static context, not a true decision-time feature. Named limitation of this slice.
- Panel is unbalanced (per-client history depth differs); clients with short history are underrepresented in training.
- `2026-06` (the `_sample` month) is sealed and untouched here — run this notebook's decision date forward to `2026-05-31` for a true out-of-time final check before the capstone paper's Results section.